In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, Qwen3ForSequenceClassification
from peft import LoraConfig, get_peft_model, get_peft_model

In [ ]:
model_name = "Qwen/Qwen3-1.7B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = Qwen3ForSequenceClassification.from_pretrained(model_name, problem_type="multi_label_classification")
lora_cfg = LoraConfig(
    r=4, 
    lora_alpha=32, 
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],   # confirmed to exist in both Qwen & Llama blocks  [oai_citation:1‡Medium](https://medium.com/%40pritisagar0427/fine-tuning-the-large-language-models-llms-using-lora-e0d9cc8960cc?utm_source=chatgpt.com)
    task_type="SEQ_CLS"
)

model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

trainable params: 806,912 || all params: 1,721,385,984 || trainable%: 0.0469


In [10]:
# Define a sample text for sentiment classification
sample_text = "This movie was really good."

# Tokenize the input
inputs = tokenizer(sample_text, return_tensors="pt", padding=True, truncation=True, max_length=128)

# Move inputs to the same device as the model
device = next(model.parameters()).device
inputs = {k: v.to(device) for k, v in inputs.items()}

# Run inference
with torch.no_grad():
    outputs = model(**inputs)
    
# Get predictions
logits = outputs.logits
predictions = torch.softmax(logits, dim=1)
predicted_class = torch.argmax(predictions, dim=1).item()

# Print results
print(f"Input text: {sample_text}")
print(f"Predicted class: {predicted_class}")
print(f"Prediction probabilities: {predictions[0].tolist()}")

Input text: This movie was really good.
Predicted class: 1
Prediction probabilities: [0.03957727923989296, 0.9604227542877197]


In [35]:
import json
from datasets import Dataset

def load_jsonl(path):
        data = []
        with open(path, 'r') as f:
            for line in f:
                data.append(json.loads(line))
        return data
    
train_path = "data/sst2_train.jsonl"
test_path = "data/sst2_test.jsonl"
train_dataset = load_jsonl(train_path)
test_dataset = load_jsonl(test_path)

train_dataset = Dataset.from_list(train_dataset)[:300]
train_dataset = Dataset.from_dict(train_dataset)
test_dataset = Dataset.from_list(test_dataset)

In [ ]:
from torch.utils.data import DataLoader

def tokenize_fn(example):
    return tokenizer(
        example["sentence"], 
        truncation=True, 
        max_length=128, 
        padding="max_length",
        return_tensors="pt"
    )
    
def collate_fn(batch):
    input_ids = torch.tensor([x['input_ids'] for x in batch], dtype=torch.long)
    attention_mask = torch.tensor([x['attention_mask'] for x in batch], dtype=torch.long)
    labels = torch.tensor([x['label'] for x in batch], dtype=torch.long)
    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'labels': labels
    }

tokenizer.pad_token = tokenizer.eos_token  # Set pad token to eos token for Qwen3

train_dataset = train_dataset.map(tokenize_fn, batched=True)
test_dataset = test_dataset.map(tokenize_fn, batched=True)

batch_size = 8
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)


Map: 100%|██████████| 872/872 [00:00<00:00, 19567.67 examples/s]


In [38]:
from tqdm import tqdm
import torch
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

device = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
if torch.backends.mps.is_available():
    print("Using MPS backend")
model.to(device)
epochs = 5
optimizer = AdamW(model.parameters(), lr=2e-5)
total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

for epoch in range(epochs):
    model.train()
    total_loss = 0
    progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    for batch in progress:
        optimizer.zero_grad()
        inputs = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        outputs = model(input_ids=inputs, attention_mask=attention_mask)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        progress.set_postfix(loss=loss.item())
    print(f"Epoch {epoch+1} avg loss: {total_loss/len(train_loader):.4f}")

Using MPS backend


Epoch 1/5:   0%|          | 0/38 [00:18<?, ?it/s]


ValueError: Cannot handle batch sizes > 1 if no padding token is defined.

In [34]:
len(batch['input_ids'])

128